# Sequences: Indexing and Slicing

> ### Learning Objectives
>
> By the end of this chapter you should be able to work with:
>
> - The concept of a sequence as an ordered collection of data
> - Indexing to access elements by position, including negative indexes, and the `IndexError`
> - Slicing to extract subsequences with `start`, `stop`, and `step`, including defaults and reversal (`[::-1]`)
> - Traversing a string with a `for` loop
> - String immutability and what it means for assignment
> - Common string methods (`upper()`, `find()`, the `in` operator, `strip()`, `replace()`)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  SETUP — run this cell first.
#
#  It draws every figure used in this chapter and switches the notebook into
#  "show me every result" mode.  Everything it needs is right here: nothing to
#  install, nothing to download, no other files required.
#
#  (Curious what a figure is made of?  The drawing code is all below.)
# ══════════════════════════════════════════════════════════════════════
import io

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Polygon, Circle
from matplotlib.lines import Line2D
from IPython.display import Image, display
from IPython.core.interactiveshell import InteractiveShell

# echo the value of *every* expression in a cell, the way the Python prompt
# does — many examples in this book show several results at once
InteractiveShell.ast_node_interactivity = "all"

# ------------------------------------------------------------- drawing ---

INK = "#1a1a1a"

MUTED = "#6b7280"

FILL = "#eef2f7"

ACCENT = "#2563eb"

WARM = "#b45309"

EDGE = "#334155"

def _frame(ax, xlim, ylim, title=None):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10, color=MUTED, pad=8)

def _box(ax, xy, text, w=2.6, h=0.9, fc=FILL, ec=EDGE, fs=9, bold=False):
    x, y = xy
    ax.add_patch(FancyBboxPatch(
        (x - w / 2, y - h / 2), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        linewidth=1.3, facecolor=fc, edgecolor=ec, zorder=2))
    ax.text(x, y, text, ha="center", va="center", fontsize=fs, color=INK,
            zorder=3, fontweight="bold" if bold else "normal")
    return xy

def _diamond(ax, xy, text, w=3.0, h=1.5, fc="#fff7ed", ec=WARM, fs=9):
    x, y = xy
    ax.add_patch(Polygon(
        [(x, y + h / 2), (x + w / 2, y), (x, y - h / 2), (x - w / 2, y)],
        closed=True, linewidth=1.3, facecolor=fc, edgecolor=ec, zorder=2))
    ax.text(x, y, text, ha="center", va="center", fontsize=fs, color=INK, zorder=3)
    return xy

def _dot(ax, xy, r=0.09):
    ax.add_patch(Circle(xy, r, facecolor=EDGE, edgecolor=EDGE, zorder=4))
    return xy

def _arrow(ax, pts, label=None, label_at=0.5, label_off=(0.0, 0.18),
           color=EDGE, ha="center"):
    """Poly-line arrow through `pts` (elbow routing), head on the last segment."""
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    ax.add_line(Line2D(xs[:-1] + [xs[-1]], ys[:-1] + [ys[-1]],
                       color=color, linewidth=1.3, zorder=1,
                       solid_capstyle="round"))
    ax.annotate("", xy=pts[-1], xytext=pts[-2],
                arrowprops=dict(arrowstyle="-|>", color=color, linewidth=1.3,
                                shrinkA=0, shrinkB=0), zorder=1)
    if label:
        i = max(0, min(len(pts) - 2, int(label_at * (len(pts) - 1))))
        mx = (pts[i][0] + pts[i + 1][0]) / 2 + label_off[0]
        my = (pts[i][1] + pts[i + 1][1]) / 2 + label_off[1]
        ax.text(mx, my, label, fontsize=8, color=MUTED, ha=ha, va="center")

def _line(ax, pts, color=EDGE):
    """Poly-line with no arrowhead — for merging branches into a shared rail."""
    ax.add_line(Line2D([p[0] for p in pts], [p[1] for p in pts], color=color,
                       linewidth=1.3, zorder=1, solid_capstyle="round"))

def _cellgrid(ax, values, origin=(0, 0), cw=1.0, ch=1.0, fs=13, fc="white"):
    """A row/table of boxed cells; `values` is a list of rows."""
    x0, y0 = origin
    for r, row in enumerate(values):
        for c, v in enumerate(row):
            x = x0 + c * cw
            y = y0 - r * ch
            ax.add_patch(plt.Rectangle((x, y - ch), cw, ch, facecolor=fc,
                                       edgecolor=EDGE, linewidth=1.2, zorder=2))
            ax.text(x + cw / 2, y - ch / 2, str(v), ha="center", va="center",
                    fontsize=fs, color=INK, family="monospace", zorder=3)

def _mockwindow(ax, w, h, title, body, titlebar="#d7dde5", face="#ffffff",
                fs=9, textcolor=INK):
    """A framed window with a title bar and monospaced body lines."""
    ax.add_patch(plt.Rectangle((0, 0), w, h, facecolor=face, edgecolor=EDGE,
                               linewidth=1.2, zorder=1))
    ax.add_patch(plt.Rectangle((0, h - 0.55), w, 0.55, facecolor=titlebar,
                               edgecolor=EDGE, linewidth=1.2, zorder=2))
    ax.text(0.2, h - 0.28, title, fontsize=9, va="center", color=INK, zorder=3)
    y = h - 1.05
    for line, colour in body:
        ax.text(0.25, y, line, fontsize=fs, va="center", family="monospace",
                color=colour or textcolor, zorder=3)
        y -= 0.5

def _index_grid(ax, items, top_label, side_label, fs=13):
    n = len(items)
    _cellgrid(ax, [items], origin=(0, 1), fs=fs)
    for i in range(n):
        ax.text(i + 0.5, 1.25, str(i), ha="center", va="bottom", fontsize=10,
                color=ACCENT)
    if top_label:
        ax.text(-0.25, 1.3, top_label, ha="right", va="bottom", fontsize=9,
                color=ACCENT)
    if side_label:
        ax.text(-0.25, 0.5, side_label, ha="right", va="center", fontsize=9,
                color=ACCENT)
    _frame(ax, (-6.4, n + 0.4), (-0.4, 2.0))

def _double_diamond(ax, stage=None):
    names = ["Understand", "Design", "Implement", "Evaluate"]
    # two diamonds: centres at x=2.6 and x=7.8, half-width 2.6, half-height 2.0
    for d, cx in enumerate((2.6, 7.8)):
        left, right, top, bot = cx - 2.6, cx + 2.6, 2.0, -2.0
        for half in (0, 1):
            name = names[2 * d + half]
            tri = ([(left, 0), (cx, top), (cx, bot)] if half == 0
                   else [(cx, top), (right, 0), (cx, bot)])
            on = (name == stage)
            ax.add_patch(Polygon(tri, closed=True, zorder=1,
                                 facecolor="#b9bfc7" if on else "#eceef1",
                                 edgecolor="none"))
            tx = cx - 1.3 if half == 0 else cx + 1.3
            ax.text(tx, 0, name, ha="center", va="center", fontsize=10,
                    color=INK if on else MUTED,
                    fontweight="bold" if on else "normal", zorder=3)
        ax.add_patch(Polygon([(left, 0), (cx, top), (right, 0), (cx, bot)],
                             closed=True, facecolor="none", edgecolor=INK,
                             linewidth=2.2, zorder=2))
        ax.plot([cx, cx], [top, bot], color=INK, linewidth=1.0, zorder=2)
    for x, y in ((0.0, 0.0), (5.2, 0.0), (10.4, 0.0)):
        ax.add_patch(Circle((x, y), 0.22, facecolor="#c9ced6", edgecolor=INK,
                            linewidth=1.2, zorder=4))
    ax.plot([-1.5, -0.22], [0, 0], color=INK, linewidth=1.6, zorder=2)
    ax.plot([10.62, 11.9], [0, 0], color=INK, linewidth=1.6, zorder=2)
    ax.plot([5.2, 5.2], [-0.22, -3.1], color=INK, linewidth=1.6, zorder=2)
    ax.text(-1.7, 0, "Problem", ha="right", va="center", fontsize=10)
    ax.text(12.1, 0, "Program", ha="left", va="center", fontsize=10)
    ax.text(5.2, -3.35, "Specification", ha="center", va="top", fontsize=10)
    _frame(ax, (-4.2, 14.2), (-4.2, 2.6))

def draw_string_offsets(ax):
    """Replaces the TikZ grid in 11_Indexing_and_Slicing ("Vader")."""
    _index_grid(ax, list("Vader"), "Character offsets:",
                "String (sequence of characters):")

# ---------------------------------------------------------------- runtime ---
_SIZES = {'string_offsets': (7.2, 1.7)}
_WIDTHS = {}
_FIGURES = {}


def _render(name):
    fig, ax = plt.subplots(figsize=_SIZES.get(name, (6.4, 4.4)), dpi=110)
    globals()["draw_" + name](ax)
    fig.tight_layout(pad=0.3)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return buf.getvalue()


def show(name, width=None):
    """Display one of this chapter's figures."""
    display(Image(_FIGURES[name], width=width or _WIDTHS.get(name, 560)))


for _n in ['string_offsets']:
    _FIGURES[_n] = _render(_n)

print("Setup complete \u2014 1 figure(s) ready.")


### Sequences

A *sequence* is a compound data type consisting of one or more pieces of data in a specific linear ordering.  An example is a string --- it is a sequence of characters.  Python defines two important operators called *indexing* and *slicing* that can be applied to sequences, including strings.

Later, we shall find out that lists and arrays in Python are also examples of sequences, and we'll be able to apply the indexing and slicing operations we learn here to those data types as well.

### Indexing

Each data item in a sequence has a position.  We denote that position using an integer which we call an *offset*.   The item at the beginning of a sequence has offset 0.  The second item in a sequence has offset 1, the third has offset 2, and so on.  In general, if an item is $n$ positions to the right of the first position, it has offset $n$.  This is why we call it an *offset*.  The second item of a sequence is offset by 1 position from the first position.  The fifth item in a sequence has offset 4, because it is offset by 4 positions from the first position --- if you start at the first position and move right 4 times, you'll be at the fifth item.

You can also think of it this way:  if an item is the $i$-th item in a sequence, it has offset $i-1$.  The following picture shows that the string `"Vader"` can be viewed as a sequence of characters with offsets 0 through 4:

In [ ]:
show("string_offsets")

In Python, you can access an item in a sequence using its offset.  This is done by putting the offset of the desired item inside a pair of square brackets after the sequence.  The square brackets are the *indexing* operator.

### Strings as Sequences

Strings are not like integers, floats, and booleans.  A string
is a **sequence**, which means it is
an ordered collection of other values.  In this chapter you'll see
how to access the characters that make up a string, and you'll
learn about some of the methods strings provide.

A string is a sequence of characters.   You can access the characters one at a time with the indexing operator:

In [ ]:
jedi = "Yoda"
letter = jedi[1]

The second statement selects character number 1 from `jedi` and assigns it to `letter`.

The expression in brackets is called an **index**.
The index indicates which character in the sequence you
want (hence the name).

But you might not get what you expect:

In [ ]:
letter

For most people, the first letter of "Yoda" is `Y`, not
`o`.  But for computer scientists, the index is an offset from the
beginning of the string, and the offset of the first letter is zero.

In [ ]:
letter = jedi[0]
letter

So `Y` is the 0th letter ("zero-eth") of "Yoda", `o` is the 1th letter ("one-eth"), and `d` is the 2th letter
("two-eth").

As an index you can use an expression that contains variables and
operators:

In [ ]:
i = 1
jedi[i]
jedi[i+1]

But the value of the index has to be an integer.  Otherwise you
get:

In [ ]:
letter = jedi[1.4]

### Slicing

Since Python strings are sequences, we can use indexing to access specific characters within a string.  This works with both string literals and string variables:

In [ ]:
"Vader"[3]       # Get 4th character from literal
s = "Skywalker"  # Make s refer to "Skywalker"
s[0]             # Get first character from string s
s[4]             # Get fifth character from string s
c = s[8]         # Get 9th character from s, give it the name c
print(c)         # print c (the 9th character from s).
print(s[2])      # print the third character of s
s[0]+s[2]+s[4]   # Concatenate the 1st, 3rd, and 5th characters
x = 7
s[x]             # Offset 7 since x refers to 7
s[x+1]           # Offset 8 since x refers to 7

In each example above, the indexing operator obtains the character from the string at the given offset.  Then we can do what we want with it:  assign a variable name to it, pass it to a function, use it in an expression, etc.  Also note that offsets can be integer literals, variables that refer to integers, or integer-valued expressions.

#### Offsets from the End

It is also possible to specify an offset from the **end** of the sequence using negative integers. Offset $-1$ is the offset of the last character.  Offset $-2$ is the offset of the second-last character, offset $-3$ is the offset of the third-last character etc..  Examples:

In [ ]:
t = "TARDIS"  # Make t refer to "TARDIS"
t[-1]         # Access the last character of t
t[-3]         # Access the third-last character of t

#### Invalid Offsets

If you use an offset that does not exist or is of the wrong type, Python will issue an error.  Remember that **offsets must be integers**.  Positive offsets of a string `s` must be between `0` and `len(s)-1`, while negative offsets must be be between `-len(s)` and `-1`.  Here are some of the things that can go wrong if you use an offset that is out of range:

In [ ]:
s = "Ice King"  # Make s refer to "Ice King"

In [ ]:
s[9]            # Offset out of range

In [ ]:
s[-10]          # Offset out of range

In [ ]:
s[5.0]          # Floats cannot be offsets. Ever.

### Slicing

*Slicing* is the act of selecting zero or more items of a sequence and forming them into a new sequence.  Slicing is similar to indexing but it allows us to specify multiple offsets at once using a convenient syntax.  The result of slicing is a new sequence consisting of the items at the specified offsets.

We can specify a contiguous range of offsets using the `:` operator --- this is the *slicing* operator.  If we write `$x$:$y$`, where $x$ and $y$ are integer expressions, this means the range of offsets between $x$ and $y-1$.  That's right, $y-1$.  The range of offsets is **inclusive** on the lower end and **exclusive** on the upper end.  Thus, `0:42` actually specifies the range of offsets $0, 1, 2,\ldots, 41$.  Notice that 42 is not included!

We can use the slicing operator to specify multiple offsets for the indexing operator to obtain substrings of a string in Python:

In [ ]:
s = "Skywalker"
t = s[3:9]       # get the substring of s between offsets 3 and 8
print (t)
print (s[0:3])   # get the substring of s between offsets 0 and 2

The exclusion of the item at the upper offset of the slicing operator in the resulting sequence probably seems strange now, but it's actually quite convenient.  For example, if `s` is a string, then `s[x:len(s)]` extracts the substring beginning at offset `x` and ending at offset `len(s)-1`, which is the last valid offset.

#### Slicing with a Non-Unit Step Size

You can select every second, third, or $n$-th item between the start and end indices by specifying a second colon and a third integer:

In [ ]:
s[0:len(s):2]    # every other character in s
s[2:7:3]         # every third character between offsets

The third integer is called the *step size* for the slicing operation.

#### Slicing with Invalid Offsets

Providing an invalid offset when indexing results in an error, as we have seen.   However,
providing an invalid offset as the starting or ending offset of a slicing operation does **not** result in an error.  The slicing operator includes in the resulting sequence all of the original sequence items that occupy valid offsets within the specified range.  Invalid offsets within the specified range are ignored.  Moreover, nonsensical slicing where the starting offset is to the right of the ending offset results in an empty sequence.

In [ ]:
s[5:25]    # valid offsets between 5 and 24 (i.e. 5 through 8)
s[-55:-5]  # valid offsets between 55th last and 6th last offset.
s[5:3]     # nonsense results in an empty sequence

Note that in the second example, offset -5 is excluded because the ending offset is always excluded when slicing.

### Traversal with a `for` loop

A lot of computations involve processing a string one character at a
time.  Often they start at the beginning, select each character in
turn, do something to it, and continue until the end.  This pattern of
processing is called a **traversal**.  One way to write a traversal
is with a `while` loop:

In [ ]:
pilot = "Chewie"
index = 0
while index < len(pilot):
    letter = pilot[index]
    print(letter)
    index = index + 1

This loop traverses the string and displays each letter on a line by
itself.  The loop condition is `index < len(pilot)`, so
when `index` is equal to the length of the string, the
condition is false, and the body of the loop doesn't run.  The
last character accessed is the one with the index `len(pilot)-1`,
which is the last character in the string.

As an exercise, write a function that takes a string as an argument
and displays the letters backward, one per line.

Another way to write a traversal is with a `for` loop:

In [ ]:
for letter in pilot:
    print(letter)

Each time through the loop, the next character in the string is assigned
to the variable `letter`.  The loop continues until no characters are
left.

### Strings are immutable

It is tempting to use the `[]` operator on the left side of an
assignment, with the intention of changing a character in a string.
For example:

In [ ]:
greeting = "So be it Jedi!"
greeting[0] = "J"

The "object" in this case is the string and the "item" is
the character you tried to assign  The reason for the error is that
strings are **immutable**, which means you can't change an
existing string.  The best you can do is create a new string
that is a variation on the original:

In [ ]:
greeting = "So be it Jedi!"
new_greeting = "T" + greeting[1:]
new_greeting

This example concatenates a new first letter onto
a slice of `greeting`.  It has no effect on
the original string.

### Searching

What does the following function do?

In [ ]:
def find(word, letter):
    index = 0
    while index < len(word):
        if word[index] == letter:
            return index
        index = index + 1
    return -1

In a sense, `find` is the inverse of the `[]` operator.
Instead of taking an index and extracting the corresponding character,
it takes a character and finds the index where that character
appears.  If the character is not found, the function returns `-1`.

This is the first example we have seen of a `return` statement
inside a loop.  If `word[index] == letter`, the function breaks
out of the loop and returns immediately.

If the character doesn't appear in the string, the program
exits the loop normally and  returns `-1`.

This pattern of computation---traversing a sequence and returning
when we find what we are looking for---is called a **search**.

As an exercise, modify `find` so that it has a
third parameter, the index in `word` where it should start
looking.

### Looping and counting

The following program counts the number of times the letter `a`
appears in a string:

In [ ]:
word = "rebellion"
count = 0
for letter in word:
    if letter == 'l':
        count = count + 1
print(count)

This program demonstrates another pattern of computation called a **counter**.  The variable `count` is initialized to 0 and then
incremented each time an `a` is found.
When the loop exits, `count`
contains the result---the total number of `l`'s.

As an exercise, encapsulate this code in a function named `count`, and generalize it so that it accepts the string and the
letter as arguments.

Then rewrite the function so that instead of
traversing the string, it uses the three-parameter version of `find` from the previous section.

### String methods

Strings provide methods that perform a variety of useful operations.
A method is similar to a function---it takes arguments and
returns a value---but the syntax is different.  For example, the
method `upper` takes a string and returns a new string with
all uppercase letters.

Instead of the function syntax `upper(word)`, it uses
the method syntax `word.upper()`.

In [ ]:
word = "rebellion"
new_word = word.upper()
new_word

This form of dot notation specifies the name of the method, `upper`, and the name of the string to apply the method to, `word`.  The empty parentheses indicate that this method takes no
arguments.

A method call is called an **invocation**; in this case, we would
say that we are invoking `upper` on `word`.

As it turns out, there is a string method named `find` that
is remarkably similar to the function we wrote:

In [ ]:
word = "rebellion"
index = word.find("e")
index

In this example, we invoke `find` on `word` and pass
the letter we are looking for as a parameter.

Actually, the `find` method is more general than our function;
it can find substrings, not just characters:

In [ ]:
word.find("be")

By default, `find` starts at the beginning of the string, but
it can take a second argument, the index where it should start:

In [ ]:
word.find("el", 3)

This is an example of an **optional argument**;
`find` can
also take a third argument, the index where it should stop:

In [ ]:
name = "bob"
name.find("b", 1, 2)

This search fails because `b` does not
appear in the index range from `1` to `2`, not including `2`.  Searching up to, but not including, the second index makes
`find` consistent with the slice operator.

### The `in` operator

The word `in` is a boolean operator that takes two strings and
returns `True` if the first appears as a substring in the second:

In [ ]:
"b" in "rebellion"
"q" in "rebellion"

For example, the following function prints all the
letters from `word1` that also appear in `word2`:

In [ ]:
def in_both(word1, word2):
    for letter in word1:
        if letter in word2:
            print(letter)

### String comparison

The relational operators work on strings.  To see if two strings are equal:

In [ ]:
if word == "rebellion":
    print("For the rebellion!")

Other relational operations are useful for putting words in alphabetical
order:

In [ ]:
if word < "rebellion":
    print("Your word, " + word + ", comes before rebellion.")
elif word > "rebellion":
    print("Your word, " + word + ", comes after rebellion.")
else:
    print("For the rebellion!")

Python does not handle uppercase and lowercase letters the same way
people do.  All the uppercase letters come before all the
lowercase letters, so:

Output:
```text
Your word, empire, comes before rebellion.
```

A common way to address this problem is to convert strings to a
standard format, such as all lowercase, before performing the
comparison.